# Figure 4 — ISC: Language vs MD regions during naturalistic movie viewing

Compares inter-subject correlation (ISC) during Figures and Life movies
in language regions (Fedorenko atlas) vs multiple-demand (MD) regions.
Each dot = one subject × one run.

**This figure requires external ISC data not produced by the main GLM pipeline.**

Configure `models/naturalistic.yaml` → `isc_data_dir` to point to the folder
containing per-subject ISC NIfTI files. If not configured, this notebook
prints a skip message and exits cleanly.

**Inputs** (from env vars injected by `invoke run-notebooks`):
- `OUTPUT_DATA_DIR` — pipeline output root
- `SOURCE_DATA_DIR` — pipeline source root (for Fedorenko + MD atlases)
- `NATURALISTIC_DATA_DIR` — path to ISC outputs (optional; skips if absent)


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from nilearn.image import resample_to_img

OUTPUT_DIR = Path(os.environ.get('OUTPUT_DATA_DIR', 'output_data'))
SOURCE_DIR = Path(os.environ.get('SOURCE_DATA_DIR', 'source_data'))
ISC_DIR    = os.environ.get('NATURALISTIC_DATA_DIR', '')

FIGURES_OUTPUT_DIR = Path(os.environ.get('FIGURES_OUTPUT_DIR',
                          str(OUTPUT_DIR / 'figures_manuscript')))
FIG_DIR = FIGURES_OUTPUT_DIR / 'fig4_isc'
FIG_DIR.mkdir(parents=True, exist_ok=True)


ISC_AVAILABLE = bool(ISC_DIR)

if not ISC_AVAILABLE:
    print(
        '[SKIP] Figure 4 (ISC) skipped.\n'
        'To enable: set naturalistic_data_dir in invoke.yaml.'
    )
else:
    ISC_DIR = Path(ISC_DIR)
    print(f'ISC data   : {ISC_DIR}')
    print(f'Figure dir : {FIG_DIR}')


ISC data   : /scratch/ibilgin/movie10_isc/output
Figure dir : /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig4_isc


In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
ATLAS_NII   = SOURCE_DIR / 'fedorenko' / 'allParcels-language-SN220.nii'
ATLAS_TXT   = SOURCE_DIR / 'fedorenko' / 'allParcels-language-SN220.txt'

# MD atlas — set path here or leave empty to skip MD comparison
MD_ATLAS = SOURCE_DIR / 'fedorenko' / 'allParcels-MD-HE197.nii'


MOVIES = {
    'figures': {'runs': ['1', '2'], 'label': 'Figures'},
    'life':    {'runs': ['1', '2'], 'label': 'Life'},
}

# ISC file naming convention:
#   {subject}_task-{movie}_run-{run}_space_MNI152NLin2009cAsym_stats-InterSC_desc-fwhm6Simple_statseries.nii.gz
ISC_FNAME_TEMPLATE = (
    '{subject}_task-{movie}_run-{run}'
    '_space_MNI152NLin2009cAsym_stats-InterSC_desc-fwhm6Simple_statseries.nii.gz'
)

SUBJECTS = ['sub-01', 'sub-02', 'sub-03', 'sub-05']
SUBJECT_COLORS = {
    'sub-01': '#E63946', 'sub-02': '#06A77D',
    'sub-03': '#457B9D', 'sub-05': '#F77F00',
}

In [3]:
# ---------------------------------------------------------------------------
# Atlas helpers
# ---------------------------------------------------------------------------

def load_parcel_labels():
    if not ATLAS_TXT.exists():
        print(f'[WARN] Atlas labels not found: {ATLAS_TXT}')
        return []
    with open(ATLAS_TXT) as f:
        return [l.strip() for l in f if l.strip()]

PARCEL_LABELS = load_parcel_labels()
print(f'Loaded {len(PARCEL_LABELS)} Fedorenko parcel labels')


def build_language_mask(roi_type='allLANG'):
    """
    Build a binary Fedorenko language mask.
    roi_type: 'allLANG' | 'LH' | 'RH' | parcel name
    """
    if not ATLAS_NII.exists():
        print(f'[WARN] Atlas NIfTI not found: {ATLAS_NII}')
        return None

    atlas_img  = nib.load(str(ATLAS_NII))
    atlas_data = atlas_img.get_fdata()

    if roi_type == 'allLANG':
        mask = (atlas_data > 0).astype(float)
    elif roi_type == 'LH':
        idx  = [i + 1 for i, l in enumerate(PARCEL_LABELS) if l.startswith('LH_')]
        mask = np.isin(atlas_data, idx).astype(float)
    elif roi_type == 'RH':
        idx  = [i + 1 for i, l in enumerate(PARCEL_LABELS) if l.startswith('RH_')]
        mask = np.isin(atlas_data, idx).astype(float)
    else:
        if roi_type not in PARCEL_LABELS:
            return None
        parcel_idx = PARCEL_LABELS.index(roi_type) + 1
        mask = (atlas_data == parcel_idx).astype(float)

    if mask.sum() == 0:
        return None
    return nib.Nifti1Image(mask, atlas_img.affine, atlas_img.header)


def build_md_mask():
    if not MD_ATLAS or not Path(MD_ATLAS).exists():
        return None
    img  = nib.load(str(MD_ATLAS))
    mask = (img.get_fdata() > 0).astype(float)
    return nib.Nifti1Image(mask, img.affine, img.header)

Loaded 10 Fedorenko parcel labels


In [4]:
# ---------------------------------------------------------------------------
# Data extraction
# ---------------------------------------------------------------------------

def _extract_mean(isc_img, roi_img):
    if roi_img is None:
        return np.nan
    if isc_img.shape[:3] != roi_img.shape[:3]:
        roi_img = resample_to_img(roi_img, isc_img, interpolation='nearest')
    mask = roi_img.get_fdata() > 0
    if mask.sum() == 0:
        return np.nan
    return float(np.nanmean(isc_img.get_fdata()[mask]))


def extract_isc_data(roi_type='allLANG'):
    """
    Extract ISC values for language (and optionally MD) regions.
    Returns a DataFrame with columns: subject, movie, movie_label,
    run, roi_type, isc_value, roi_name.
    """
    lang_mask = build_language_mask(roi_type)
    md_mask   = build_md_mask()

    if lang_mask is None:
        print(f'[SKIP] No language mask for roi_type={roi_type}')
        return pd.DataFrame()

    rows = []
    for subject in SUBJECTS:
        for movie, info in MOVIES.items():
            for run in info['runs']:
                fname = ISC_FNAME_TEMPLATE.format(
                    subject=subject, movie=movie, run=run)
                fpath = ISC_DIR / fname
                if not fpath.exists():
                    continue
                isc_img   = nib.load(str(fpath))
                lang_val  = _extract_mean(isc_img, lang_mask)
                md_val    = _extract_mean(isc_img, md_mask)

                if not np.isnan(lang_val):
                    rows.append({'subject': subject, 'movie': movie,
                                 'movie_label': info['label'], 'run': run,
                                 'roi_type': 'Language', 'isc_value': lang_val,
                                 'roi_name': roi_type})
                if not np.isnan(md_val):
                    rows.append({'subject': subject, 'movie': movie,
                                 'movie_label': info['label'], 'run': run,
                                 'roi_type': 'MD', 'isc_value': md_val,
                                 'roi_name': roi_type})

    return pd.DataFrame(rows)

In [5]:
# ---------------------------------------------------------------------------
# Figure builder
# ---------------------------------------------------------------------------

def make_figure4(df, roi_name='Both Hemispheres', dpi=300):
    if df is None or len(df) == 0:
        print(f'[SKIP] No ISC data for {roi_name}')
        return None

    np.random.seed(42)

    df['category'] = df['movie_label'] + ' × ' + df['roi_type']
    cat_order      = ['Figures × Language', 'Figures × MD',
                      'Life × Language',    'Life × MD']
    cats_present   = [c for c in cat_order if c in df['category'].unique()]
    x_pos          = np.arange(len(cats_present)) * 0.9

    fig, ax = plt.subplots(figsize=(10, 8))

    means = [df[df['category'] == c]['isc_value'].mean() for c in cats_present]
    sems  = [df[df['category'] == c]['isc_value'].sem()  for c in cats_present]
    colors = ['#B8D4E8' if 'Language' in c else '#E8D4C8' for c in cats_present]

    ax.bar(x_pos, means, 0.85, color=colors,
           alpha=0.85, edgecolor='#2C3E50', linewidth=1.8, zorder=2)
    ax.errorbar(x_pos, means, yerr=sems, fmt='none', color='#2C3E50',
                capsize=7, linewidth=2.5, capthick=2.5, zorder=3)

    for xi, cat in zip(x_pos, cats_present):
        for _, row in df[df['category'] == cat].iterrows():
            jitter = np.random.uniform(-0.15, 0.15)
            ax.scatter(xi + jitter, row['isc_value'],
                       color=SUBJECT_COLORS.get(row['subject'], '#7F8C8D'),
                       s=200, zorder=10, edgecolor='white', linewidth=2.5, alpha=0.95)

    ax.set_ylabel('ISC values', fontsize=16, fontweight='600',
                  color='#2C3E50', labelpad=12)
    ax.set_xticks(x_pos)
    xlabels = []
    for c in cats_present:
        movie = 'Figures' if 'Figures' in c else 'Life'
        rtype = 'Language' if 'Language' in c else 'MD'
        xlabels.append(f'{movie}\n{rtype}')
    ax.set_xticklabels(xlabels, fontsize=14, color='#34495E', fontweight='600')
    ax.set_ylim([0, 0.35])
    ax.set_xlim([x_pos[0] - 0.6, x_pos[-1] + 0.6])
    ax.grid(axis='y', alpha=0.25, linestyle='--', linewidth=1, color='#BDC3C7')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axhline(0, color='#7F8C8D', linewidth=1.5, linestyle='--', alpha=0.6, zorder=1)
    ax.tick_params(labelsize=12, colors='#34495E', width=1.5, length=6)

    fig.text(0.5, 0.97, 'Functional Selectivity during naturalistic stimuli',
             ha='center', fontsize=20, fontweight='bold', color='black')
    fig.text(0.5, 0.93, f'{roi_name} — each dot is a subject and a run',
             ha='center', fontsize=14, style='italic', color='#566573')

    subjects = sorted(df['subject'].unique())
    handles  = [plt.scatter([], [], s=200, color=SUBJECT_COLORS.get(s, '#7F8C8D'),
                             edgecolor='white', linewidth=2.5, alpha=0.95)
                for s in subjects]
    ax.legend(handles, subjects, loc='upper right', frameon=True, fontsize=12,
              fancybox=True, facecolor='white', edgecolor='#BDC3C7')

    plt.tight_layout(rect=[0, 0.02, 1, 0.90])

    safe     = roi_name.replace(' ', '_').replace('/', '-')
    out_path = FIG_DIR / f'fig4_isc_{safe}.png'
    out_pdf  = FIG_DIR / f'fig4_isc_{safe}.pdf'
    fig.savefig(str(out_path), dpi=dpi, bbox_inches='tight', facecolor='white')
    fig.savefig(str(out_pdf),  bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  Saved: {out_path.name}')
    return out_path

In [6]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
if not ISC_AVAILABLE:
    print('[SKIP] No ISC data configured — skipping figure generation.')
else:
    roi_configs = [
        ('allLANG', 'Both Hemispheres'),
        ('LH',      'Left Hemisphere'),
        ('RH',      'Right Hemisphere'),
    ] + [(p, p) for p in PARCEL_LABELS]

    for roi_type, roi_name in roi_configs:
        print(f'\nExtracting ISC: {roi_name} ...')
        df = extract_isc_data(roi_type=roi_type)
        if len(df) > 0:
            safe     = roi_type.replace(' ', '_').replace('/', '-')
            csv_path = FIG_DIR / f'data_fig4_{safe}.csv'
            df.to_csv(csv_path, index=False)
            make_figure4(df, roi_name=roi_name)

    print(f'\nFigures saved to: {FIG_DIR}')


Extracting ISC: Both Hemispheres ...


  Saved: fig4_isc_Both_Hemispheres.png

Extracting ISC: Left Hemisphere ...


  Saved: fig4_isc_Left_Hemisphere.png

Extracting ISC: Right Hemisphere ...


  Saved: fig4_isc_Right_Hemisphere.png

Extracting ISC: LH_IFGorb ...


  Saved: fig4_isc_LH_IFGorb.png

Extracting ISC: LH_IFG ...


...

  Saved: fig4_isc_LH_PostTemp.png

Extracting ISC: RH_IFGorb ...
[SKIP] No language mask for roi_type=RH_IFGorb

Extracting ISC: RH_IFG ...


  Saved: fig4_isc_RH_IFG.png

Extracting ISC: RH_MFG ...


  Saved: fig4_isc_RH_MFG.png

Extracting ISC: RH_AntTemp ...


  Saved: fig4_isc_RH_AntTemp.png

Extracting ISC: RH_PostTemp ...


  Saved: fig4_isc_RH_PostTemp.png

Figures saved to: /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig4_isc
